# Notebook 00: Project Setup and FastF1 Sanity Check

The goal of this notebook is the following:

1. Set up folder paths and FastF1 cache
2. Load one session successfully
3. Inspect key tables (results + laps)
4. Save sample data to disk


## 0) One-time installs

Make sure to install the dependencies by installing requirements.txt

```
pip install -r requirements.txt
```


In [2]:
# Core Python and data libraries
from pathlib import Path
import pandas as pd
import numpy as np

# FastF1 imports
import fastf1

print('Libraries imported successfully.')

Libraries imported successfully.


In [3]:
# Resolve project root whether notebook runs from repo root or notebooks folder
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

data_dir = project_root / 'data'
raw_dir = data_dir / 'raw'
processed_dir = data_dir / 'processed'
cache_dir = raw_dir / 'fastf1_cache'

for folder in [data_dir, raw_dir, processed_dir, cache_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# Enable local cache so repeated API calls are much faster
fastf1.Cache.enable_cache(str(cache_dir))

print('Project root:', project_root)
print('FastF1 cache:', cache_dir)

Project root: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction
FastF1 cache: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\fastf1_cache


## 1) Load one real session (sanity check)

Use a recent event so data is available and easier to inspect.

You can change these values later.


In [8]:
season = 2026
gp_name = 'Australia'
session_type = 'Q'  # Q=Qualifying, R=Race, FP1/FP2/FP3=Practice

session = fastf1.get_session(season, gp_name, session_type)

# laps=True loads lap-level data. telemetry=False keeps this first run faster.
session.load(laps=True, telemetry=False, weather=True, messages=False)

print('Loaded session:', session.event['EventName'], session_type)
print('Session date:', session.date)
print('Drivers found:', len(session.drivers))

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 18
core        WARNING 	No lap data for driver 55
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 55)
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '6', '16', '81', '1', '44', '30', '41', '5', '27', '87', '31', '10', '23', '43', '14',

Loaded session: Australian Grand Prix Q
Session date: 2026-03-07 05:00:00
Drivers found: 22


In [6]:
# inspect classification/result table
results_df = session.results.copy()

candidate_cols = [
    'Abbreviation', 'FullName', 'TeamName', 'Position',
    'Q1', 'Q2', 'Q3', 'GridPosition', 'Status', 'Points'
]
existing_cols = [c for c in candidate_cols if c in results_df.columns]

display(results_df[existing_cols].head(20))
print('Results shape:', results_df.shape)

,Abbreviation,FullName,TeamName,Position,Q1,Q2,Q3,GridPosition,Status,Points
63,RUS,George Russell,Mercedes,1.0,0 days 00:01:19.507000,0 days 00:01:18.934000,0 days 00:01:18.518000,NaN,,NaN
12,ANT,Kimi Antonelli,Mercedes,2.0,0 days 00:01:20.120000,0 days 00:01:19.435000,0 days 00:01:18.811000,NaN,,NaN
6,HAD,Isack Hadjar,Red Bull Racing,3.0,0 days 00:01:20.023000,0 days 00:01:19.653000,0 days 00:01:19.303000,NaN,,NaN
16,LEC,Charles Leclerc,Ferrari,4.0,0 days 00:01:20.226000,0 days 00:01:19.357000,0 days 00:01:19.327000,NaN,,NaN
81,PIA,Oscar Piastri,McLaren,5.0,0 days 00:01:19.664000,0 days 00:01:19.525000,0 days 00:01:19.380000,NaN,,NaN
1,NOR,Lando Norris,McLaren,6.0,0 days 00:01:20.010000,0 days 00:01:19.882000,0 days 00:01:19.475000,NaN,,NaN
44,HAM,Lewis Hamilton,Ferrari,7.0,0 days 00:01:19.811000,0 days 00:01:19.921000,0 days 00:01:19.478000,NaN,,NaN
30,LAW,Liam Lawson,Racing Bulls,8.0,0 days 00:01:20.491000,0 days 00:01:20.144000,0 days 00:01:19.994000,NaN,,NaN
41,LIN,Arvid Lindblad,Racing Bulls,9.0,0 days 00:01:20.409000,0 days 00:01:19.971000,0 days 00:01:21.247000,NaN,,NaN
5,BOR,Gabriel Bortoleto,Audi,10.0,0 days 00:01:20.495000,0 days 00:01:20.221000,NaT,NaN,,NaN


Results shape: (22, 22)


In [7]:
# inspect lap table 
laps_df = session.laps.copy()

lap_cols = [
    'Driver', 'DriverNumber', 'LapNumber', 'LapTime',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'Compound', 'TyreLife', 'Stint',
    'IsPersonalBest', 'TrackStatus', 'PitOutTime', 'PitInTime'
]
existing_lap_cols = [c for c in lap_cols if c in laps_df.columns]

display(laps_df[existing_lap_cols].head(20))
print('Laps shape:', laps_df.shape)

,Driver,DriverNumber,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,Compound,TyreLife,Stint,IsPersonalBest,TrackStatus,PitOutTime,PitInTime
0,RUS,63,1.0,0 days 00:01:51.876000,0 days 00:00:41.997000,0 days 00:00:19.351000,0 days 00:00:50.528000,SOFT,1.0,1.0,False,1,0 days 00:16:28.955000,NaT
1,RUS,63,2.0,0 days 00:01:19.840000,0 days 00:00:27.990000,0 days 00:00:17.599000,0 days 00:00:34.251000,SOFT,2.0,1.0,True,1,NaT,NaT
2,RUS,63,3.0,0 days 00:02:01.569000,0 days 00:00:40.873000,0 days 00:00:22.025000,0 days 00:00:58.671000,SOFT,3.0,1.0,False,1,NaT,0 days 00:21:24.695000
3,RUS,63,4.0,NaT,0 days 00:00:46.779000,0 days 00:00:22.489000,NaT,SOFT,4.0,2.0,False,125,0 days 00:21:42.023000,0 days 00:23:38.517000
4,RUS,63,5.0,0 days 00:01:50.313000,0 days 00:00:52.390000,0 days 00:00:19.564000,0 days 00:00:38.359000,SOFT,5.0,3.0,False,1,0 days 00:32:12.182000,NaT
5,RUS,63,6.0,0 days 00:01:33.246000,0 days 00:00:31.116000,0 days 00:00:23.100000,0 days 00:00:39.030000,SOFT,6.0,3.0,False,1,NaT,NaT
6,RUS,63,7.0,0 days 00:01:19.507000,0 days 00:00:27.908000,0 days 00:00:17.413000,0 days 00:00:34.186000,SOFT,7.0,3.0,True,1,NaT,NaT
7,RUS,63,8.0,NaT,0 days 00:00:44.215000,0 days 00:00:22.959000,NaT,SOFT,8.0,3.0,False,12,NaT,0 days 00:38:31.168000
8,RUS,63,9.0,0 days 00:02:19.248000,0 days 00:01:01.667000,0 days 00:00:23.062000,0 days 00:00:54.519000,SOFT,1.0,4.0,False,1,0 days 00:48:36.384000,NaT
9,RUS,63,10.0,0 days 00:01:18.934000,0 days 00:00:27.816000,0 days 00:00:17.287000,0 days 00:00:33.831000,SOFT,2.0,4.0,True,1,NaT,NaT


Laps shape: (353, 31)


In [8]:
# save sample outputs so we have tangible artifacts from notebook 00
event_name_safe = str(session.event['EventName']).replace(' ', '_')
base_name = f"{season}_{event_name_safe}_{session_type}"

results_out = raw_dir / f"{base_name}_results.csv"
laps_out = raw_dir / f"{base_name}_laps.parquet"

results_df.to_csv(results_out, index=False)
laps_df.to_parquet(laps_out, index=False)

print('Saved:', results_out)
print('Saved:', laps_out)

Saved: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\2025_Bahrain_Grand_Prix_Q_results.csv
Saved: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\2025_Bahrain_Grand_Prix_Q_laps.parquet
